<a href="https://colab.research.google.com/github/aydanali/ECON3916-Stats-and-ML/blob/main/Lab12/StatsML_Lab12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.tools.eval_measures import rmse
import matplotlib.pyplot as plt

# Step 1: Ingestion from external source
url = 'https://raw.githubusercontent.com/aydanali/ECON3916-Stats-and-ML/refs/heads/main/Data/Zillow_ZHVI_2026_Micro.csv'
df = pd.read_csv(url)

In [18]:
df.head()

,Home_Value,Square_Footage,Property_Age,Distance_to_Transit,School_District_Rating
0,329705.74,1941.0,5.5,6.45,Excellent
1,183343.63,1364.3,35.2,2.15,Average
2,354551.73,2386.9,52.4,0.75,Good
3,325773.17,2192.1,50.2,5.25,Excellent
4,359743.12,3069.8,66.5,12.69,Excellent


In [5]:
# Step 2: Defining the formula
# Utilizing the R-style patsy formula interface allows for elegant, readable model specification
formula = 'Home_Value ~ Square_Footage + Property_Age + Distance_to_Transit + School_District_Rating'

In [19]:
# Step 3: Fitting the model and printing the summary
model = smf.ols(formula=formula, data=df)
results = model.fit()
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:             Home_Value   R-squared:                       0.766
Model:                            OLS   Adj. R-squared:                  0.765
Method:                 Least Squares   F-statistic:                     542.5
Date:                Mon, 16 Mar 2026   Prob (F-statistic):          2.81e-309
Time:                        20:17:42   Log-Likelihood:                -12072.
No. Observations:                1000   AIC:                         2.416e+04
Df Residuals:                     993   BIC:                         2.419e+04
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                          coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
In

In [20]:
# Step 4: Generating predictions
# We extract the predicted values vector to transition from explanation to prediction
y_pred = results.predict(df)

In [21]:
y_pred

,0
0,312288.586866
1,223813.440910
2,331610.316284
3,307402.426806
4,392714.851722
...,...
995,406437.277604
996,380041.214086
997,236618.275309
998,315508.480776


In [22]:
# Step 5: Calculate RMSE between the actuals and the predictions
model_rmse = rmse(df['Home_Value'], y_pred)
print(f"\nThe Predictive RMSE is: ${model_rmse:,.2f}")


The Predictive RMSE is: $42,316.69


In [23]:
"""
residual_forensics.py
─────────────────────────────────────────────────────────────────────────────
Hedonic Pricing OLS — Interactive Residual Forensics Dashboard
Built on: Home_Value ~ Square_Footage + Property_Age + Distance_to_Transit
          + School_District_Rating (one-hot encoded)
─────────────────────────────────────────────────────────────────────────────
"""

import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.express as px

# ─── 0. YOUR DATAFRAME ───────────────────────────────────────────────────────
# Paste / load your full df here. The 5-row sample below matches your schema —
# replace with pd.read_csv(...) or your existing df object.
# df = pd.DataFrame({
#     "Home_Value":            [329705.74, 183343.63, 354551.73, 325773.17, 359743.12],
#     "Square_Footage":        [1941.0,    1364.3,    2386.9,    2192.1,    3069.8],
#     "Property_Age":          [5.5,       35.2,      52.4,      50.2,      66.5],
#     "Distance_to_Transit":   [6.45,      2.15,      0.75,      5.25,      12.69],
#     "School_District_Rating":["Excellent","Average", "Good",   "Excellent","Excellent"],
# })

# ─── 1. ENCODE CATEGORICAL FEATURE ───────────────────────────────────────────
# School_District_Rating is ordinal — one-hot encode it and drop one level
# (drop_first=True) to avoid multicollinearity / dummy variable trap.
# pd.get_dummies returns a DataFrame of 0/1 indicator columns.
rating_dummies = pd.get_dummies(
    df["School_District_Rating"],
    prefix="SDR",
    drop_first=True,    # drops the alphabetically first level ("Average") as baseline
    dtype=float,        # keeps dtype consistent with numeric features
)

# ─── 2. BUILD FEATURE MATRIX ─────────────────────────────────────────────────
# Concatenate the three numeric predictors with the dummy columns, then add an
# explicit intercept column (statsmodels does NOT add one by default).
X_raw = df[["Square_Footage", "Property_Age", "Distance_to_Transit"]]
X     = sm.add_constant(pd.concat([X_raw, rating_dummies], axis=1))

y = df["Home_Value"]   # dependent variable: raw price in dollars

# ─── 3. FIT OLS ──────────────────────────────────────────────────────────────
model  = sm.OLS(y, X)
result = model.fit()

print(result.summary())   # inspect R², coefficients, p-values

# ─── 4. EXTRACT FITTED VALUES & RESIDUALS ────────────────────────────────────
# result.fittedvalues → Series of ŷ, same index as y
# result.resid        → Series of ε = y − ŷ, same index as y
# Both are attributes cached on the results object after .fit() — not methods.
fitted    = result.fittedvalues
residuals = result.resid

# ─── 5. RMSE ─────────────────────────────────────────────────────────────────
rmse = np.sqrt(np.mean(residuals ** 2))
print(f"\nRMSE: ${rmse:,.2f}")

# ─── 6. OUTLIER DETECTION: |ε| > 2σ ─────────────────────────────────────────
resid_std    = residuals.std()
outlier_mask = np.abs(residuals) > 2 * resid_std

# Human-readable label for Plotly's color channel
point_labels = outlier_mask.map({
    True:  "Outlier (|ε| > 2σ)",
    False: "Normal",
})

# ─── 7. ASSEMBLE PLOT DATAFRAME ──────────────────────────────────────────────
plot_df = pd.DataFrame({
    "Fitted Values (ŷ)":        fitted,
    "Residuals (ε)":             residuals,
    "Status":                    point_labels,
    # Surface raw features in hover tooltip for forensic context
    "Square_Footage":            df["Square_Footage"],
    "Property_Age":              df["Property_Age"],
    "Distance_to_Transit (mi)":  df["Distance_to_Transit"],
    "School_District_Rating":    df["School_District_Rating"],
    "Actual Home Value":         y.map("${:,.0f}".format),
})

# ─── 8. COLOR MAP ────────────────────────────────────────────────────────────
color_map = {
    "Normal":              "#5B8DB8",   # steel blue
    "Outlier (|ε| > 2σ)": "#C0152A",   # stark crimson
}

# ─── 9. SCATTER PLOT ─────────────────────────────────────────────────────────
# px.scatter is the current non-deprecated API.
# 'color' maps the categorical Status column → color_discrete_map entries.
fig = px.scatter(
    plot_df,
    x="Fitted Values (ŷ)",
    y="Residuals (ε)",
    color="Status",
    color_discrete_map=color_map,
    hover_data={
        "Square_Footage":           True,
        "Property_Age":             True,
        "Distance_to_Transit (mi)": True,
        "School_District_Rating":   True,
        "Actual Home Value":        True,
        "Status":                   False,  # already encoded by color
    },
    opacity=0.78,
    template="plotly_dark",
    labels={
        "Fitted Values (ŷ)": "Fitted Values  ŷ  ($)",
        "Residuals (ε)":      "Residuals  ε = y − ŷ  ($)",
    },
)

# ─── 10. REFERENCE LINES ─────────────────────────────────────────────────────
# Zero line — OLS residuals should be centered here if model is well-specified
fig.add_hline(
    y=0,
    line_dash="dash",
    line_color="rgba(255,255,255,0.45)",
    line_width=1.5,
    annotation_text="ε = 0",
    annotation_position="bottom right",
    annotation_font_color="rgba(255,255,255,0.45)",
)

# ±2σ threshold lines — marks the outlier detection boundary
for sign, label, pos in [(+1, "+2σ", "top right"), (-1, "−2σ", "bottom right")]:
    fig.add_hline(
        y=sign * 2 * resid_std,
        line_dash="dot",
        line_color="rgba(192, 21, 42, 0.5)",
        line_width=1.2,
        annotation_text=label,
        annotation_position=pos,
        annotation_font_color="rgba(192, 21, 42, 0.85)",
    )

# ─── 11. LAYOUT ──────────────────────────────────────────────────────────────
n_outliers = outlier_mask.sum()

fig.update_layout(
    title={
        "text": (
            f"Residual Forensics — Hedonic Pricing OLS<br>"
            f"<sup>RMSE: ${rmse:,.2f}  |  "
            f"Outliers: {n_outliers} / {len(df)}  "
            f"(|ε| > 2σ = ${2*resid_std:,.2f})</sup>"
        ),
        "x": 0.5,
        "xanchor": "center",
        "font": {"size": 17},
    },
    legend={
        "title": "Residual Status",
        "bgcolor": "rgba(0,0,0,0.3)",
        "bordercolor": "rgba(255,255,255,0.1)",
        "borderwidth": 1,
    },
    xaxis={
        "showgrid": True,
        "gridcolor": "rgba(255,255,255,0.07)",
        "tickprefix": "$",
        "tickformat": ",.0f",
    },
    yaxis={
        "showgrid": True,
        "gridcolor": "rgba(255,255,255,0.07)",
        "tickprefix": "$",
        "tickformat": ",.0f",
    },
    margin={"t": 100, "b": 60, "l": 80, "r": 40},
    font={"family": "monospace"},
)

fig.update_traces(
    marker={"size": 8, "line": {"width": 0.6, "color": "rgba(255,255,255,0.25)"}}
)

# ─── 12. RENDER ──────────────────────────────────────────────────────────────
fig.show()
# To save: fig.write_html("residual_forensics_dashboard.html")

                            OLS Regression Results                            
Dep. Variable:             Home_Value   R-squared:                       0.766
Model:                            OLS   Adj. R-squared:                  0.765
Method:                 Least Squares   F-statistic:                     542.5
Date:                Mon, 16 Mar 2026   Prob (F-statistic):          2.81e-309
Time:                        20:17:50   Log-Likelihood:                -12072.
No. Observations:                1000   AIC:                         2.416e+04
Df Residuals:                     993   BIC:                         2.419e+04
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                8.984e+04   6